# RNN training — 15-frame non-freeze windows, offset-0 anchor, full event-plus-60 targets

This notebook trains the same InT-style checkpoint model using the outputs produced by `rnn_data_prep.ipynb`.

## Local folder layout

Both notebooks are expected to live in:

```text
~/Downloads/Meta-control/
├── rnn_data_prep.ipynb
├── rnn_training.ipynb
├── physics_abstraction_master/
├── rnn_training_data/
├── rnn_testing_data/
├── compressed_frame_cache_100x128/
└── ball_position_cache/
```

Training results are written to:

```text
~/Downloads/Meta-control/rnn_training_results/
```

The paths and cache-ID convention in this notebook exactly match `rnn_data_prep.ipynb`.

Key sampling rules:

- Each training sample uses a random consecutive **15-frame input window**.
- The entire input window must be before the first frozen event frame. If the first goal/ground event is at frame 200, the latest valid input window is frames 185–199.
- The target begins at **offset 0**, the ball position at the last input frame, and continues through every remaining frame of the scene, including the 60 frozen post-event frames.
- Testing uses the first 15 frames of each held-out scene and predicts through that scene’s final frozen frame.
- Training runs for **150 epochs** on GPU.
- After every epoch, the notebook immediately overwrites the best-model checkpoint whenever held-out total loss improves. Final predictions and overlays are generated from that best epoch, not automatically from epoch 150.


In [ ]:
# ============================================================
# 0. GPU environment check
# ============================================================

import sys, os, subprocess

print("Python executable:", sys.executable)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

try:
    import torch
    print("torch version:", torch.__version__)
    print("torch cuda version:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Could not import torch:", repr(e))
    raise


In [ ]:
# ============================================================
# 1. Imports and settings
# ============================================================

from pathlib import Path
from collections import OrderedDict
import random
import json
import os
import shutil
import time
from contextlib import nullcontext

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

# ----------------------------
# Paths
# ----------------------------
HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"

TRAIN_DATA_ROOT = BASE_DIR / "rnn_training_data"
TRAIN_EXP_FOLDERS = [
    "five_containers",
    "four_containers",
    "one_line",
]
EXPECTED_TRAIN_SCENES_BY_FOLDER = {
    "five_containers": 2900,
    "four_containers": 2500,
    "one_line": 600,
}
EXPECTED_TRAIN_SCENES = sum(EXPECTED_TRAIN_SCENES_BY_FOLDER.values())

TEST_DATA_ROOT = BASE_DIR / "rnn_testing_data"
TEST_EXP_FOLDERS = ["exp1", "exp2"]
EXPECTED_TEST_SCENES = 106

# Dedicated output root for model checkpoints, predictions, and diagnostics.
TRIAL_ROOT = BASE_DIR / "rnn_training_results"

FRAME_CACHE_DIR = BASE_DIR / "compressed_frame_cache_100x128"
BALL_CACHE_DIR = BASE_DIR / "ball_position_cache"

if not BASE_DIR.exists():
    raise FileNotFoundError(
        f"Missing workspace: {BASE_DIR}\n"
        "Expected this notebook and its prepared data under "
        "~/Downloads/Meta-control."
    )

# The prepared data/cache folders are validated in the next cell.
TRIAL_ROOT.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Reproducibility / data settings
# ----------------------------
SEED = 7
MAX_FRAMES_PER_TRAIN_SCENE = 1000

# Each scene contributes this many random valid windows per epoch.
N_HISTORY = 15
TRAIN_RANDOM_WINDOW_PER_SCENE_PER_EPOCH = True
WINDOWS_PER_SCENE_PER_EPOCH = 20

# Testing/overlay evaluation uses the first N_HISTORY frames only.
TEST_USE_FIRST_HISTORY_ONLY = True
TEST_WINDOW_STRIDE = 15  # record-keeping only; start=0 is used for testing.

CHECKPOINT_STRIDE = 10
OVERLAY_STRIDE = 10

ORIGINAL_FRAME_WIDTH = 800.0
ORIGINAL_FRAME_HEIGHT = 1024.0
COORD_SCALE = np.array(
    [ORIGINAL_FRAME_WIDTH, ORIGINAL_FRAME_HEIGHT],
    dtype=np.float32,
)
ERR_MAG_PIXEL_SCALE_FOR_PLOTS = float(np.mean(COORD_SCALE))

IMAGE_W = 100
IMAGE_H = 128

# Training settings: unchanged except EPOCHS=150.
BATCH_SIZE = 32
EPOCHS = 150
LR = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = min(4, os.cpu_count() or 1)

HIDDEN_CHANNELS = 96
TIME_EMBED_DIM = 32
LAMBDA_ERR = 0.2

# Save one overlay for every held-out scene.
N_OVERLAY_PLOTS = EXPECTED_TEST_SCENES

# The data-preparation notebook produces every required cache.
ALLOW_BUILD_MISSING_TRAIN_CACHES = False
ALLOW_BUILD_MISSING_TEST_CACHES = False

# Limit retained memory maps per DataLoader worker to avoid opening thousands
# of cache files simultaneously. This does not change sampling or model inputs.
FRAME_MEMMAP_LRU_SIZE = 64

REQUIRE_GPU = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if REQUIRE_GPU and DEVICE != "cuda":
    raise RuntimeError(
        "GPU was requested, but torch.cuda.is_available() is False. "
        "Check the first cell: either the Jupyter job has no GPU, "
        "or torch is CPU-only."
    )

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

USE_AMP = DEVICE == "cuda"

def autocast_context():
    return (
        torch.amp.autocast("cuda", enabled=True)
        if USE_AMP
        else nullcontext()
    )

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Results root:", TRIAL_ROOT.resolve())
print("Offset-0 current-position target included in normal position loss: True")
print("Training data root:", TRAIN_DATA_ROOT.resolve())
print("Testing data root:", TEST_DATA_ROOT.resolve())
print("Ball-position cache directory:", BALL_CACHE_DIR.resolve())
print("Compressed-frame cache directory:", FRAME_CACHE_DIR.resolve())
print("EXPECTED_TRAIN_SCENES:", EXPECTED_TRAIN_SCENES)
print("EXPECTED_TEST_SCENES:", EXPECTED_TEST_SCENES)
print("EPOCHS:", EPOCHS)
print("LAMBDA_ERR:", LAMBDA_ERR)
print("N_HISTORY:", N_HISTORY)
print(
    "TRAIN_RANDOM_WINDOW_PER_SCENE_PER_EPOCH:",
    TRAIN_RANDOM_WINDOW_PER_SCENE_PER_EPOCH,
)
print("WINDOWS_PER_SCENE_PER_EPOCH:", WINDOWS_PER_SCENE_PER_EPOCH)
print("TEST_WINDOW_STRIDE:", TEST_WINDOW_STRIDE)
print("IMAGE_H, IMAGE_W:", IMAGE_H, IMAGE_W)
print("NUM_WORKERS:", NUM_WORKERS)


In [ ]:
# ============================================================
# 1b. Preflight checks for the prepared datasets
# ============================================================

required_roots = [
    TRAIN_DATA_ROOT,
    TEST_DATA_ROOT,
    FRAME_CACHE_DIR,
    BALL_CACHE_DIR,
]
missing_roots = [path for path in required_roots if not path.exists()]
if missing_roots:
    raise FileNotFoundError(
        "Missing required prepared data/cache folders:\n"
        + "\n".join(f"  {path}" for path in missing_roots)
    )

train_summary_path = TRAIN_DATA_ROOT / "generation_summary.json"
test_summary_path = TEST_DATA_ROOT / "testing_generation_summary.json"

if train_summary_path.exists():
    with open(train_summary_path, "r") as f:
        train_generation_summary = json.load(f)
    print("Training generation summary accepted_total:",
          train_generation_summary.get("accepted_total"))
    print("Training generation summary accepted_by_setting:",
          train_generation_summary.get("accepted_by_setting"))
else:
    print("WARNING: training generation_summary.json was not found; "
          "the scene scan in the next cell remains authoritative.")

if test_summary_path.exists():
    with open(test_summary_path, "r") as f:
        test_generation_summary = json.load(f)
    print("Testing generation summary generated_total:",
          test_generation_summary.get("generated_total"))
    print("Testing generation summary generated_scene_counts:",
          test_generation_summary.get("generated_scene_counts"))
else:
    print("WARNING: testing_generation_summary.json was not found; "
          "the scene scan in the next cell remains authoritative.")

print("Preflight roots found.")


In [ ]:
# ============================================================
# 2. Find train/test scenes and read event metadata
# ============================================================

def load_scene_metadata(scene_dir):
    metadata_path = Path(scene_dir) / "metadata.json"
    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing metadata.json: {metadata_path}")
    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    required = {"output_frames", "stop_event_frame"}
    missing = required - set(metadata)
    if missing:
        raise RuntimeError(
            f"{metadata_path} is missing required fields: {sorted(missing)}"
        )
    return metadata


def _count_png_frames(scene_dir):
    files = sorted((Path(scene_dir) / "frames").glob("frame_*.png"))
    if len(files) == 0:
        files = sorted((Path(scene_dir) / "frames").glob("*.png"))
    return len(files)


def find_training_scene_dirs(
    data_root=TRAIN_DATA_ROOT,
    exp_folders=TRAIN_EXP_FOLDERS,
    max_frames=MAX_FRAMES_PER_TRAIN_SCENE,
):
    """
    Training scenes do not need high-resolution frames/. They must have:
      - metadata.json with output_frames and stop_event_frame;
      - at least N_HISTORY non-frozen frames (indices 0..stop_event_frame-1);
      - output length <= max_frames.
    """
    scene_dirs = []
    skipped_too_short_nonfreeze = []
    skipped_too_long = []
    folder_counts = {}

    for exp in exp_folders:
        exp_dir = Path(data_root) / exp
        if not exp_dir.exists():
            raise FileNotFoundError(f"Missing training folder: {exp_dir}")

        folder_scenes = []
        for scene_dir in sorted(exp_dir.iterdir()):
            if (
                not scene_dir.is_dir()
                or scene_dir.name.startswith("_")
                or scene_dir.name.startswith(".")
                or not (scene_dir / "metadata.json").exists()
            ):
                continue

            metadata = load_scene_metadata(scene_dir)
            n_frames = int(metadata["output_frames"])
            stop_event_frame = int(metadata["stop_event_frame"])
            nonfreeze_frames = stop_event_frame  # frames 0..event-1

            if nonfreeze_frames < N_HISTORY:
                skipped_too_short_nonfreeze.append(
                    (scene_dir, nonfreeze_frames, stop_event_frame, n_frames)
                )
                continue
            if n_frames > max_frames:
                skipped_too_long.append((scene_dir, n_frames))
                continue

            folder_scenes.append(scene_dir)
            scene_dirs.append(scene_dir)

        folder_counts[exp] = len(folder_scenes)

    print("Training usable by folder:", folder_counts)
    print("Training scenes usable:", len(scene_dirs))
    print(
        "Training skipped with fewer than "
        f"{N_HISTORY} non-freeze frames:",
        len(skipped_too_short_nonfreeze),
    )
    print(f"Training skipped > {max_frames} frames:", len(skipped_too_long))

    if folder_counts != EXPECTED_TRAIN_SCENES_BY_FOLDER:
        raise RuntimeError(
            "Training folder counts do not match the completed 6,000-scene set.\n"
            f"Expected: {EXPECTED_TRAIN_SCENES_BY_FOLDER}\n"
            f"Found:    {folder_counts}"
        )
    if len(scene_dirs) != EXPECTED_TRAIN_SCENES:
        raise RuntimeError(
            f"Expected {EXPECTED_TRAIN_SCENES} usable training scenes, "
            f"found {len(scene_dirs)}."
        )

    return (
        scene_dirs,
        skipped_too_short_nonfreeze,
        skipped_too_long,
        folder_counts,
    )


def find_testing_scene_dirs(
    data_root=TEST_DATA_ROOT,
    exp_folders=TEST_EXP_FOLDERS,
):
    """
    Testing scenes must have high-resolution frames/ for overlays and at least
    N_HISTORY natural, non-frozen frames before the first goal/ground event.
    """
    scene_dirs = []
    skipped_too_short_nonfreeze = []
    folder_counts = {}

    for exp in exp_folders:
        exp_dir = Path(data_root) / exp
        if not exp_dir.exists():
            raise FileNotFoundError(f"Missing testing folder: {exp_dir}")

        folder_scenes = []
        for scene_dir in sorted(exp_dir.iterdir()):
            if (
                not scene_dir.is_dir()
                or not (scene_dir / "metadata.json").exists()
                or not (scene_dir / "frames").exists()
            ):
                continue

            metadata = load_scene_metadata(scene_dir)
            n_frames = int(metadata["output_frames"])
            stop_event_frame = int(metadata["stop_event_frame"])
            nonfreeze_frames = stop_event_frame

            if nonfreeze_frames < N_HISTORY:
                skipped_too_short_nonfreeze.append(
                    (scene_dir, nonfreeze_frames, stop_event_frame, n_frames)
                )
                continue

            png_count = _count_png_frames(scene_dir)
            if png_count != n_frames:
                raise RuntimeError(
                    f"{scene_dir}: metadata reports {n_frames} frames, "
                    f"but frames/ contains {png_count} PNGs."
                )

            folder_scenes.append(scene_dir)
            scene_dirs.append(scene_dir)

        folder_counts[exp] = len(folder_scenes)

    print("Testing usable by folder:", folder_counts)
    print("Testing scenes usable:", len(scene_dirs))
    print(
        "Testing skipped with fewer than "
        f"{N_HISTORY} non-freeze frames:",
        len(skipped_too_short_nonfreeze),
    )

    if len(scene_dirs) != EXPECTED_TEST_SCENES:
        raise RuntimeError(
            f"Expected all {EXPECTED_TEST_SCENES} testing scenes to be usable, "
            f"found {len(scene_dirs)}. Skipped: {skipped_too_short_nonfreeze[:10]}"
        )

    return scene_dirs, skipped_too_short_nonfreeze, folder_counts


(
    train_scene_dirs,
    train_skipped_short,
    train_skipped_long,
    train_folder_counts,
) = find_training_scene_dirs()

(
    test_scene_dirs,
    test_skipped_short,
    test_folder_counts,
) = find_testing_scene_dirs()

if len(train_scene_dirs) == 0:
    raise RuntimeError("No training scenes found.")
if len(test_scene_dirs) == 0:
    raise RuntimeError("No testing scenes found.")

all_scene_dirs_for_cache = train_scene_dirs + test_scene_dirs

print("\nFirst few train scenes:")
for path in train_scene_dirs[:10]:
    print(" ", path)

print("\nFirst few test scenes:")
for path in test_scene_dirs[:10]:
    print(" ", path)

split_info = {
    "train_scenes": [str(path) for path in train_scene_dirs],
    "test_scenes": [str(path) for path in test_scene_dirs],
    "train_scene_count": len(train_scene_dirs),
    "test_scene_count": len(test_scene_dirs),
    "train_folder_counts": train_folder_counts,
    "test_folder_counts": test_folder_counts,
    "train_data_root": str(TRAIN_DATA_ROOT),
    "test_data_root": str(TEST_DATA_ROOT),
    "n_history": N_HISTORY,
    "input_window_constraint": (
        "all N_HISTORY input frames must have indices below stop_event_frame"
    ),
    "train_max_frames": MAX_FRAMES_PER_TRAIN_SCENE,
    "train_skipped_too_short_nonfreeze": [
        {
            "scene": str(path),
            "nonfreeze_frames": int(nonfreeze_frames),
            "stop_event_frame": int(stop_event_frame),
            "output_frames": int(output_frames),
        }
        for path, nonfreeze_frames, stop_event_frame, output_frames
        in train_skipped_short
    ],
    "train_skipped_too_long": [
        {"scene": str(path), "frames": int(n)}
        for path, n in train_skipped_long
    ],
    "test_skipped_too_short_nonfreeze": [
        {
            "scene": str(path),
            "nonfreeze_frames": int(nonfreeze_frames),
            "stop_event_frame": int(stop_event_frame),
            "output_frames": int(output_frames),
        }
        for path, nonfreeze_frames, stop_event_frame, output_frames
        in test_skipped_short
    ],
}
with open(TRIAL_ROOT / "scene_split.json", "w") as f:
    json.dump(split_info, f, indent=2)


In [ ]:
# ============================================================
# 3. Cache helpers and cache validation
# ============================================================
# Both the 6,000 training scenes and 106 testing scenes use caches under:
#   ~/Downloads/Meta-control/compressed_frame_cache_100x128
#   ~/Downloads/Meta-control/ball_position_cache
# Cache IDs are relative to BASE_DIR, matching rnn_data_prep.ipynb.
# ============================================================

def scene_rel_for_cache(scene_dir):
    p = Path(scene_dir)
    try:
        return p.resolve().relative_to(BASE_DIR.resolve())
    except Exception:
        return p


def safe_scene_id(scene_dir):
    rel = scene_rel_for_cache(scene_dir)
    return (
        str(rel)
        .replace("/", "__")
        .replace("\\", "__")
        .replace(":", "")
    )


def ball_cache_path(scene_dir):
    return BALL_CACHE_DIR / (
        f"{safe_scene_id(scene_dir)}__ball_positions_from_frames.csv"
    )


def compressed_frame_cache_path(scene_dir):
    return FRAME_CACHE_DIR / (
        f"{safe_scene_id(scene_dir)}"
        f"__frames_{IMAGE_H}x{IMAGE_W}_rgb_uint8.npy"
    )


def scene_is_training(scene_dir):
    value = str(scene_rel_for_cache(scene_dir))
    return (
        value.startswith("rnn_training_data/")
        or value == "rnn_training_data"
    )


def get_frame_files(scene_dir):
    frames_dir = Path(scene_dir) / "frames"
    files = sorted(frames_dir.glob("frame_*.png"))
    if len(files) == 0:
        files = sorted(frames_dir.glob("*.png"))
    return files


def build_ball_position_cache_from_scene(scene_dir):
    out_path = ball_cache_path(scene_dir)
    csv_path = Path(scene_dir) / "simulation_dataset.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing simulation_dataset.csv: {csv_path}")

    df = pd.read_csv(csv_path)
    required = {"ball_x", "ball_y"}
    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(f"{csv_path} is missing columns: {missing}")

    if "frame" in df.columns:
        frame_index = df["frame"].astype(int).to_numpy()
    else:
        frame_index = np.arange(len(df), dtype=int)

    if "frame_path" in df.columns:
        frame_path = df["frame_path"].astype(str).to_numpy()
    elif "frame_file" in df.columns:
        frame_path = [
            str(Path(scene_dir) / "frames" / name)
            for name in df["frame_file"].astype(str).to_numpy()
        ]
    else:
        frame_path = [
            str(Path(scene_dir) / "frames" / f"frame_{i:04d}.png")
            for i in frame_index
        ]

    out_df = pd.DataFrame(
        {
            "frame_index": frame_index,
            "frame_path": frame_path,
            "ball_x": df["ball_x"].astype(float).to_numpy(),
            "ball_y": df["ball_y"].astype(float).to_numpy(),
        }
    )
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_path, index=False)
    return out_path


def build_compressed_frame_cache_from_scene(scene_dir):
    out_path = compressed_frame_cache_path(scene_dir)
    frame_files = get_frame_files(scene_dir)
    if len(frame_files) == 0:
        raise FileNotFoundError(
            f"No PNG frames found in {Path(scene_dir) / 'frames'}"
        )

    arrs = []
    resample = (
        Image.Resampling.BILINEAR
        if hasattr(Image, "Resampling")
        else Image.BILINEAR
    )
    for frame_path in frame_files:
        image = Image.open(frame_path).convert("RGB")
        image = image.resize((IMAGE_W, IMAGE_H), resample)
        array = np.asarray(image, dtype=np.uint8)
        arrs.append(np.transpose(array, (2, 0, 1)))

    frames = np.stack(arrs, axis=0)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(out_path, frames)
    return out_path


def load_cached_ball_positions(scene_dir):
    cache_path = ball_cache_path(scene_dir)
    if not cache_path.exists():
        if scene_is_training(scene_dir) and not ALLOW_BUILD_MISSING_TRAIN_CACHES:
            raise FileNotFoundError(
                "Missing training ball-position cache for scene:\n"
                f"  {scene_dir}\nExpected:\n  {cache_path}"
            )
        if (not scene_is_training(scene_dir)) and not ALLOW_BUILD_MISSING_TEST_CACHES:
            raise FileNotFoundError(
                f"Missing testing ball-position cache: {cache_path}"
            )
        print("Building missing ball-position cache:", cache_path)
        build_ball_position_cache_from_scene(scene_dir)

    df = pd.read_csv(cache_path)
    required = {"frame_index", "frame_path", "ball_x", "ball_y"}
    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(
            f"Cached label file {cache_path} is missing columns: {missing}"
        )
    return df


def load_compressed_frame_cache(scene_dir):
    cache_path = compressed_frame_cache_path(scene_dir)
    if not cache_path.exists():
        if scene_is_training(scene_dir) and not ALLOW_BUILD_MISSING_TRAIN_CACHES:
            raise FileNotFoundError(
                "Missing training compressed-frame cache for scene:\n"
                f"  {scene_dir}\nExpected:\n  {cache_path}"
            )
        if (not scene_is_training(scene_dir)) and not ALLOW_BUILD_MISSING_TEST_CACHES:
            raise FileNotFoundError(
                f"Missing testing compressed-frame cache: {cache_path}"
            )
        print("Building missing compressed-frame cache:", cache_path)
        build_compressed_frame_cache_from_scene(scene_dir)
    return cache_path


def check_or_build_caches(scene_dirs, label):
    rows = []
    for index, scene_dir in enumerate(scene_dirs, start=1):
        ball_path = ball_cache_path(scene_dir)
        frame_path = compressed_frame_cache_path(scene_dir)

        try:
            metadata = load_scene_metadata(scene_dir)
            expected_frames = int(metadata["output_frames"])

            positions = load_cached_ball_positions(scene_dir)
            cache_file = load_compressed_frame_cache(scene_dir)
            frames = np.load(cache_file, mmap_mode="r")

            shape_ok = (
                frames.ndim == 4
                and frames.shape[1:] == (3, IMAGE_H, IMAGE_W)
                and frames.dtype == np.uint8
            )
            if not shape_ok:
                raise RuntimeError(
                    f"Bad frame cache shape/dtype for {cache_file}: "
                    f"shape={frames.shape}, dtype={frames.dtype}"
                )
            if len(positions) != expected_frames or frames.shape[0] != expected_frames:
                raise RuntimeError(
                    f"{scene_dir}: metadata={expected_frames}, "
                    f"ball cache={len(positions)}, frame cache={frames.shape[0]}"
                )

            expected_indices = np.arange(expected_frames, dtype=int)
            actual_indices = positions["frame_index"].to_numpy(dtype=int)
            if not np.array_equal(actual_indices, expected_indices):
                raise RuntimeError(
                    f"{ball_path}: frame_index is not consecutive from zero."
                )

            rows.append(
                {
                    "scene": str(scene_dir),
                    "ball_cache": str(ball_path),
                    "frame_cache": str(frame_path),
                    "status": "ok",
                    "shape": str(frames.shape),
                    "stop_event_frame": int(metadata["stop_event_frame"]),
                    "output_frames": expected_frames,
                }
            )
        except Exception as error:
            rows.append(
                {
                    "scene": str(scene_dir),
                    "ball_cache": str(ball_path),
                    "frame_cache": str(frame_path),
                    "status": "failed",
                    "error": repr(error),
                }
            )
            raise

        if index % 500 == 0 or index == len(scene_dirs):
            print(f"{label}: checked {index}/{len(scene_dirs)} scenes")

    df = pd.DataFrame(rows)
    output_path = TRIAL_ROOT / f"cache_check_{label}.csv"
    df.to_csv(output_path, index=False)
    print(f"{label}: checked {len(df)} scenes; saved {output_path}")
    return df


print("Checking training caches...")
train_cache_check_df = check_or_build_caches(train_scene_dirs, "train")
print("Checking testing caches...")
test_cache_check_df = check_or_build_caches(test_scene_dirs, "test")


In [ ]:
# ============================================================
# 4. Dataset builder:
# random 15-frame non-freeze windows -> offset 0 + every point to scene end
# ============================================================

CHECKPOINT_MODE = "offset0_everypoint_sliding15_nonfreeze_input"


def everypoint_offsets_from_start(T, start, n_history=N_HISTORY):
    """
    Offsets 0..scene end measured from the last input frame.

    Offset 0 is the current position at the last observed frame. The final
    offset always targets frame T-1, including the post-event frozen phase.
    """
    input_last_idx = int(start) + int(n_history) - 1
    max_future_offset = int(T) - input_last_idx - 1
    if max_future_offset < 0:
        return []
    return list(range(0, max_future_offset + 1))


def max_valid_nonfreeze_window_start(
    stop_event_frame,
    n_history=N_HISTORY,
):
    """
    The event frame is the first frozen frame. Therefore the input window must
    end no later than stop_event_frame - 1:

        start + n_history - 1 <= stop_event_frame - 1
        start <= stop_event_frame - n_history
    """
    return int(stop_event_frame) - int(n_history)


def summarize_sliding_window_plan(scene_dirs):
    rows = []
    max_checkpoints = 0
    max_future_offset = 1
    total_train_random_samples_per_epoch = 0
    total_test_windows = 0

    train_scene_set = {str(path) for path in train_scene_dirs}
    test_scene_set = {str(path) for path in test_scene_dirs}

    for scene_dir in scene_dirs:
        scene_key = str(scene_dir)
        metadata = load_scene_metadata(scene_dir)
        stop_event_frame = int(metadata["stop_event_frame"])

        pos_df = (
            load_cached_ball_positions(scene_dir)
            .dropna(subset=["ball_x", "ball_y"])
            .reset_index(drop=True)
        )
        cache_path = load_compressed_frame_cache(scene_dir)
        cached_shape = np.load(cache_path, mmap_mode="r").shape
        T = min(len(pos_df), int(cached_shape[0]))

        max_start = max_valid_nonfreeze_window_start(stop_event_frame)
        train_usable = max_start >= 0
        test_starts = [0] if max_start >= 0 else []

        max_offset_from_start0 = int(T) - N_HISTORY
        n_checkpoints_from_start0 = (
            max_offset_from_start0 + 1
            if train_usable
            else 0
        )

        if train_usable:
            max_checkpoints = max(
                max_checkpoints,
                n_checkpoints_from_start0,
            )
            max_future_offset = max(
                max_future_offset,
                max_offset_from_start0,
            )

        if scene_key in train_scene_set and train_usable:
            total_train_random_samples_per_epoch += int(
                WINDOWS_PER_SCENE_PER_EPOCH
            )
        if scene_key in test_scene_set:
            total_test_windows += len(test_starts)

        rows.append(
            {
                "scene": scene_key,
                "frames": int(T),
                "stop_event_frame": stop_event_frame,
                "nonfreeze_frames_before_event": stop_event_frame,
                "max_valid_nonfreeze_window_start": int(max_start),
                "latest_valid_input_first_frame": int(max_start),
                "latest_valid_input_last_frame": int(
                    max_start + N_HISTORY - 1
                ) if max_start >= 0 else None,
                "max_future_offset_from_start0": int(
                    max_offset_from_start0
                ),
                "max_checkpoints_from_start0_including_offset0": int(
                    n_checkpoints_from_start0
                ),
                "n_train_samples_per_epoch": int(
                    WINDOWS_PER_SCENE_PER_EPOCH
                    if scene_key in train_scene_set and train_usable
                    else 0
                ),
                "n_test_windows_first_history_only": int(
                    len(test_starts)
                    if scene_key in test_scene_set
                    else 0
                ),
                "input_window_entirely_nonfreeze": True,
                "target_continues_through_scene_end": True,
                "offset0_included": True,
                "test_use_first_history_only": bool(
                    TEST_USE_FIRST_HISTORY_ONLY
                ),
                "test_window_stride": int(TEST_WINDOW_STRIDE),
            }
        )

    if max_checkpoints <= 0:
        raise RuntimeError(
            "Sliding-window checkpoint plan failed: no valid targets."
        )

    plan_df = pd.DataFrame(rows)
    return (
        plan_df,
        int(max_checkpoints),
        int(max_future_offset),
        int(total_train_random_samples_per_epoch),
        int(total_test_windows),
    )


class RandomSlidingWindowDataset(Dataset):
    """
    Scene-balanced training dataset.

    Each scene appears WINDOWS_PER_SCENE_PER_EPOCH times per epoch. Each access
    samples a random 15-frame input window wholly before stop_event_frame, then
    targets offset 0 through the final frame of the variable-length cache.
    """

    def __init__(
        self,
        scene_dirs,
        max_checkpoints,
        global_max_future_offset,
    ):
        self.scene_dirs = list(scene_dirs)
        self.max_checkpoints = int(max_checkpoints)
        self.global_max_future_offset = float(
            max(1, global_max_future_offset)
        )

        self.samples = []
        self.scene_tables = {}
        self.cache_paths = {}
        self.frame_arrays = OrderedDict()
        self.max_start_by_scene = {}
        self.n_frames_by_scene = {}
        self.stop_event_by_scene = {}

        for scene_dir in self.scene_dirs:
            scene_key = str(scene_dir)
            metadata = load_scene_metadata(scene_dir)
            stop_event_frame = int(metadata["stop_event_frame"])

            pos_df = load_cached_ball_positions(scene_dir)
            pos_df = (
                pos_df
                .dropna(subset=["ball_x", "ball_y"])
                .reset_index(drop=True)
            )

            cache_path = load_compressed_frame_cache(scene_dir)
            cached_shape = np.load(cache_path, mmap_mode="r").shape
            T = min(len(pos_df), int(cached_shape[0]))
            pos_df = pos_df.iloc[:T].reset_index(drop=True)

            max_start = max_valid_nonfreeze_window_start(
                stop_event_frame
            )
            if max_start < 0:
                continue
            if stop_event_frame >= T:
                raise RuntimeError(
                    f"{scene_key}: stop_event_frame={stop_event_frame} "
                    f"is outside T={T}."
                )

            self.scene_tables[scene_key] = pos_df
            self.cache_paths[scene_key] = cache_path
            self.max_start_by_scene[scene_key] = int(max_start)
            self.n_frames_by_scene[scene_key] = int(T)
            self.stop_event_by_scene[scene_key] = stop_event_frame
            self.samples.append(scene_key)

        if len(self.samples) == 0:
            raise RuntimeError(
                "No random sliding-window training samples created."
            )

    def __len__(self):
        return (
            len(self.samples)
            * int(WINDOWS_PER_SCENE_PER_EPOCH)
        )

    @staticmethod
    def _close_memmap(array):
        mmap = getattr(array, "_mmap", None)
        if mmap is not None:
            try:
                mmap.close()
            except Exception:
                pass

    def _get_frames(self, scene_key):
        if scene_key in self.frame_arrays:
            array = self.frame_arrays.pop(scene_key)
            self.frame_arrays[scene_key] = array
            return array

        array = np.load(
            self.cache_paths[scene_key],
            mmap_mode="r",
        )
        self.frame_arrays[scene_key] = array

        while len(self.frame_arrays) > FRAME_MEMMAP_LRU_SIZE:
            _, old_array = self.frame_arrays.popitem(last=False)
            self._close_memmap(old_array)

        return array

    def __getitem__(self, idx):
        scene_idx = (
            int(idx)
            // int(WINDOWS_PER_SCENE_PER_EPOCH)
        )
        scene_idx %= len(self.samples)

        scene_key = self.samples[scene_idx]
        max_start = self.max_start_by_scene[scene_key]
        start = random.randint(0, max_start)
        return self._build_item(scene_key, start, idx)

    def _build_item(self, scene_key, start, idx):
        frames = self._get_frames(scene_key)
        pos_df = self.scene_tables[scene_key]
        T = self.n_frames_by_scene[scene_key]
        stop_event_frame = self.stop_event_by_scene[scene_key]

        start = int(start)
        input_last_idx = start + N_HISTORY - 1

        if input_last_idx >= stop_event_frame:
            raise RuntimeError(
                f"Input window entered frozen phase: scene={scene_key}, "
                f"start={start}, input_last_idx={input_last_idx}, "
                f"stop_event_frame={stop_event_frame}"
            )

        offsets = everypoint_offsets_from_start(T, start)
        if len(offsets) == 0:
            raise RuntimeError(
                f"No target offsets for scene={scene_key}, "
                f"start={start}, T={T}"
            )

        X_np = (
            np.asarray(
                frames[start:start + N_HISTORY],
                dtype=np.float32,
            )
            / 255.0
        )
        if X_np.shape[0] != N_HISTORY:
            raise RuntimeError(
                f"Bad input length for {scene_key}: {X_np.shape}"
            )

        Y_np = np.zeros(
            (self.max_checkpoints, 2),
            dtype=np.float32,
        )
        mask_np = np.zeros(
            self.max_checkpoints,
            dtype=np.float32,
        )
        offset_np = np.zeros(
            self.max_checkpoints,
            dtype=np.float32,
        )
        time_np = np.zeros(
            self.max_checkpoints,
            dtype=np.float32,
        )

        for checkpoint_index, offset in enumerate(
            offsets[:self.max_checkpoints]
        ):
            target_idx = input_last_idx + int(offset)
            xy_pix = (
                pos_df.iloc[target_idx][["ball_x", "ball_y"]]
                .to_numpy(dtype=np.float32)
            )
            Y_np[checkpoint_index] = xy_pix / COORD_SCALE
            mask_np[checkpoint_index] = 1.0
            offset_np[checkpoint_index] = float(offset)
            time_np[checkpoint_index] = (
                float(offset)
                / self.global_max_future_offset
            )

        return (
            torch.from_numpy(X_np),
            torch.from_numpy(Y_np),
            torch.from_numpy(mask_np),
            torch.from_numpy(offset_np),
            torch.from_numpy(time_np),
            idx,
        )

    def get_meta_for_scene_start(self, scene_key, start):
        pos_df = self.scene_tables[scene_key]
        T = self.n_frames_by_scene[scene_key]
        stop_event_frame = self.stop_event_by_scene[scene_key]
        input_last_idx = int(start) + N_HISTORY - 1
        offsets = everypoint_offsets_from_start(T, int(start))

        return {
            "scene": scene_key,
            "start": int(start),
            "input_last_frame": int(
                pos_df.iloc[input_last_idx]["frame_index"]
            ),
            "stop_event_frame": int(stop_event_frame),
            "final_frame": int(
                pos_df.iloc[T - 1]["frame_index"]
            ),
            "n_frames": int(T),
            "n_checkpoints": int(len(offsets)),
            "checkpoint_offsets": [
                int(value) for value in offsets
            ],
            "input_window_entirely_nonfreeze": bool(
                input_last_idx < stop_event_frame
            ),
        }


class DeterministicSlidingWindowDataset(
    RandomSlidingWindowDataset
):
    """
    Held-out evaluation uses start=0: the first 15 non-frozen frames predict
    offset 0 through the final frozen frame.
    """

    def __init__(
        self,
        scene_dirs,
        max_checkpoints,
        global_max_future_offset,
        window_stride=TEST_WINDOW_STRIDE,
    ):
        super().__init__(
            scene_dirs,
            max_checkpoints,
            global_max_future_offset,
        )
        self.window_stride = int(window_stride)
        self.window_samples = []
        self.metadata = []

        for scene_key in self.samples:
            start = 0
            if N_HISTORY - 1 >= self.stop_event_by_scene[scene_key]:
                raise RuntimeError(
                    f"Testing first-{N_HISTORY} window enters frozen phase: "
                    f"{scene_key}"
                )
            self.window_samples.append((scene_key, start))
            self.metadata.append(
                self.get_meta_for_scene_start(scene_key, start)
            )

        if len(self.window_samples) == 0:
            raise RuntimeError(
                "No deterministic sliding-window test samples created."
            )

    def __len__(self):
        return len(self.window_samples)

    def __getitem__(self, idx):
        scene_key, start = self.window_samples[idx]
        return self._build_item(scene_key, start, idx)

    def get_meta(self, idx):
        return self.metadata[idx]


SlidingWindowCheckpointDataset = RandomSlidingWindowDataset


def make_loaders_for_sliding_windows():
    (
        checkpoint_plan_df,
        max_checkpoints,
        global_max_future_offset,
        n_train_samples_per_epoch,
        n_test_windows,
    ) = summarize_sliding_window_plan(
        all_scene_dirs_for_cache
    )

    checkpoint_plan_path = (
        TRIAL_ROOT
        / "checkpoint_plan_all_scenes_everypoint_sliding15_nonfreeze.csv"
    )
    checkpoint_plan_df.to_csv(
        checkpoint_plan_path,
        index=False,
    )

    print("\n" + "=" * 100)
    print("Checkpoint mode:", CHECKPOINT_MODE)
    print("Checkpoint plan saved to:", checkpoint_plan_path)
    print("MAX_CHECKPOINTS:", max_checkpoints)
    print(
        "GLOBAL_MAX_FUTURE_OFFSET:",
        global_max_future_offset,
    )
    print(
        "Training policy:",
        WINDOWS_PER_SCENE_PER_EPOCH,
        f"random consecutive {N_HISTORY}-frame windows per scene per epoch",
    )
    print(
        "Input constraint: every input frame is before stop_event_frame"
    )
    print(
        "Target policy: offset 0 at the last input frame "
        "+ every later point through the final frozen frame"
    )
    print(
        f"Testing policy: first {N_HISTORY} frames only for each test scene"
    )
    print("Train samples per epoch:", n_train_samples_per_epoch)
    print("Test windows:", n_test_windows)
    print("Scene frame-count summary:")
    display(checkpoint_plan_df["frames"].describe())
    print("Stop-event-frame summary:")
    display(checkpoint_plan_df["stop_event_frame"].describe())

    train_ds = RandomSlidingWindowDataset(
        train_scene_dirs,
        max_checkpoints=max_checkpoints,
        global_max_future_offset=global_max_future_offset,
    )
    test_ds = DeterministicSlidingWindowDataset(
        test_scene_dirs,
        max_checkpoints=max_checkpoints,
        global_max_future_offset=global_max_future_offset,
        window_stride=TEST_WINDOW_STRIDE,
    )

    loader_kwargs = {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "pin_memory": DEVICE == "cuda",
    }
    if NUM_WORKERS > 0:
        loader_kwargs["persistent_workers"] = True
        loader_kwargs["prefetch_factor"] = 2

    train_loader = DataLoader(
        train_ds,
        shuffle=True,
        **loader_kwargs,
    )
    test_loader = DataLoader(
        test_ds,
        shuffle=False,
        **loader_kwargs,
    )

    print("Train scenes:", len(train_scene_dirs))
    print("Test scenes:", len(test_scene_dirs))
    print("Train samples per epoch:", len(train_ds))
    print("Test first-history samples:", len(test_ds))
    print("Train batches:", len(train_loader))
    print("Test batches:", len(test_loader))

    batch = next(iter(train_loader))
    print("Example batch shapes:")
    print("X:", batch[0].shape)
    print("Y:", batch[1].shape)
    print("mask:", batch[2].shape)
    print("checkpoint_offsets:", batch[3].shape)
    print("checkpoint_times:", batch[4].shape)

    return (
        train_ds,
        test_ds,
        train_loader,
        test_loader,
        max_checkpoints,
        global_max_future_offset,
        checkpoint_plan_df,
    )


print("Coordinate normalization: image-size normalized")
print("ORIGINAL_FRAME_WIDTH:", ORIGINAL_FRAME_WIDTH)
print("ORIGINAL_FRAME_HEIGHT:", ORIGINAL_FRAME_HEIGHT)
print("COORD_SCALE:", COORD_SCALE)
print("N_HISTORY:", N_HISTORY)
print("CHECKPOINT_MODE:", CHECKPOINT_MODE)
print(
    "TRAIN_RANDOM_WINDOW_PER_SCENE_PER_EPOCH:",
    TRAIN_RANDOM_WINDOW_PER_SCENE_PER_EPOCH,
)
print(
    "WINDOWS_PER_SCENE_PER_EPOCH:",
    WINDOWS_PER_SCENE_PER_EPOCH,
)
print(
    "TEST_USE_FIRST_HISTORY_ONLY:",
    TEST_USE_FIRST_HISTORY_ONLY,
)
print("TEST_WINDOW_STRIDE:", TEST_WINDOW_STRIDE)


In [ ]:
# ============================================================
# 5. Model: InT-style recurrent visual encoder + attention + time-conditioned decoder
# ============================================================
# Image-only model: it receives 15 compressed RGB frames and checkpoint times,
# but no explicit ball position, velocity, or object parameters.
# ============================================================

class InTCheckpointRNN(nn.Module):
    def __init__(self, hidden_channels=96, time_embed_dim=32):
        super().__init__()

        self.hidden_channels = hidden_channels
        self.time_embed_dim = time_embed_dim

        self.input_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, hidden_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
        )

        # InT-style local recurrent/lateral update: one 3x3 recurrent kernel per channel.
        self.recurrent_conv = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=hidden_channels,
            bias=False,
        )

        self.gate_conv = nn.Conv2d(hidden_channels * 2, hidden_channels, kernel_size=1)
        self.hidden_bias = nn.Parameter(torch.zeros(1, hidden_channels, 1, 1))

        self.attn_conv = nn.Conv2d(hidden_channels, 1, kernel_size=1)

        self.scene_mlp = nn.Sequential(
            nn.Linear(hidden_channels, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.05),
            nn.Linear(192, 192),
            nn.ReLU(inplace=True),
        )

        # Time features are [tau, tau^2, sin(pi tau), cos(pi tau)].
        self.time_mlp = nn.Sequential(
            nn.Linear(4, time_embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(time_embed_dim, time_embed_dim),
            nn.ReLU(inplace=True),
        )

        decoder_in = 192 + time_embed_dim
        self.checkpoint_decoder = nn.Sequential(
            nn.Linear(decoder_in, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.05),
            nn.Linear(192, 96),
            nn.ReLU(inplace=True),
        )

        self.position_head = nn.Linear(96, 2)
        self.error_head = nn.Linear(96, 1)

    def _attention_pool(self, h):
        B, C, H, W = h.shape
        logits = self.attn_conv(h).view(B, 1, H * W)
        weights = torch.softmax(logits, dim=-1)
        h_flat = h.view(B, C, H * W)
        pooled = (h_flat * weights).sum(dim=-1)
        return pooled

    def _time_features(self, checkpoint_times):
        tau = checkpoint_times.clamp(min=0.0, max=1.5)
        return torch.stack([
            tau,
            tau ** 2,
            torch.sin(np.pi * tau),
            torch.cos(np.pi * tau),
        ], dim=-1)

    def forward(self, x, checkpoint_times, return_hidden=False):
        B, T, C, H, W = x.shape
        h = None
        hidden_trace = []

        for t in range(T):
            z_t = self.input_conv(x[:, t])
            if h is None:
                h = torch.zeros_like(z_t)

            candidate = torch.tanh(z_t + self.recurrent_conv(h) + self.hidden_bias)
            gate = torch.sigmoid(self.gate_conv(torch.cat([z_t, h], dim=1)))
            h = gate * candidate + (1.0 - gate) * h

            if return_hidden:
                hidden_trace.append(h)

        pooled = self._attention_pool(h)
        scene_code = self.scene_mlp(pooled)

        time_feat = self._time_features(checkpoint_times)
        time_code = self.time_mlp(time_feat)

        M = checkpoint_times.shape[1]
        scene_code_expanded = scene_code.unsqueeze(1).expand(B, M, scene_code.shape[-1])
        decoder_in = torch.cat([scene_code_expanded, time_code], dim=-1)
        decoded = self.checkpoint_decoder(decoder_in)

        pred_pos = self.position_head(decoded)
        pred_err_mag_raw = self.error_head(decoded)

        if return_hidden:
            hidden_trace = torch.stack(hidden_trace, dim=1)
            return pred_pos, pred_err_mag_raw, hidden_trace
        return pred_pos, pred_err_mag_raw


def positive_error_magnitude(pred_err_mag_raw):
    return nn.functional.softplus(pred_err_mag_raw).squeeze(-1)


def scene_balanced_masked_mean(per_checkpoint_values, mask, eps=1e-8):
    mask = mask.to(dtype=per_checkpoint_values.dtype)
    per_scene = (per_checkpoint_values * mask).sum(dim=1) / (mask.sum(dim=1) + eps)
    return per_scene.mean()


def augmented_loss(y_true, target_mask, pred_pos, pred_err_mag_raw, lambda_err):
    target_mask = target_mask.to(dtype=pred_pos.dtype)

    per_checkpoint_pos = ((pred_pos - y_true) ** 2).mean(dim=-1)
    pos_loss = scene_balanced_masked_mean(per_checkpoint_pos, target_mask)

    delta_norm = (y_true - pred_pos).detach()
    true_err_mag_norm = torch.linalg.norm(delta_norm, dim=-1)
    pred_err_mag_norm = positive_error_magnitude(pred_err_mag_raw)
    per_checkpoint_err = (pred_err_mag_norm - true_err_mag_norm) ** 2
    err_loss = scene_balanced_masked_mean(per_checkpoint_err, target_mask)

    total = pos_loss + lambda_err * err_loss
    return total, pos_loss, err_loss


def make_new_model_and_optimizer():
    model = InTCheckpointRNN(hidden_channels=HIDDEN_CHANNELS, time_embed_dim=TIME_EMBED_DIM).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    return model, optimizer, scaler


In [ ]:
# ============================================================
# 6. Train/evaluate/save one run
# ============================================================

def unnormalize_y(y_norm):
    return y_norm * COORD_SCALE


def plot_prediction_overlay(row_group, out_path=None):
    """
    One overlay per test scene:
      - background = last input frame
      - predicted checkpoints = colored dots with dashed uncertainty circles
      - predicted checkpoints are connected by straight line segments
      - true full trajectory is drawn on top in red
    """
    row_group = row_group.sort_values("checkpoint_index").copy()
    if len(row_group) == 0:
        return

    scene = Path(row_group.iloc[0]["scene"])
    input_last_frame = int(row_group.iloc[0]["input_last_frame"])
    final_frame = int(row_group.iloc[0]["final_frame"])

    frame_path = scene / "frames" / f"frame_{input_last_frame:04d}.png"
    if not frame_path.exists():
        frame_files = get_frame_files(scene)
        frame_path = frame_files[min(input_last_frame, len(frame_files) - 1)]

    img = Image.open(frame_path).convert("RGB")

    fig, ax = plt.subplots(figsize=(6, 7.5))
    ax.imshow(img)

    pred_xy = row_group[["pred_x", "pred_y"]].to_numpy(dtype=float)
    true_xy = row_group[["true_x", "true_y"]].to_numpy(dtype=float)
    pred_unc = row_group["predicted_uncertainty"].to_numpy(dtype=float)
    offsets = row_group["checkpoint_offset"].to_numpy(dtype=int)

    if len(pred_xy) >= 2:
        segs = np.stack([pred_xy[:-1], pred_xy[1:]], axis=1)
        lc = LineCollection(segs, linewidths=1.8, alpha=0.75)
        ax.add_collection(lc)

    colors = plt.cm.viridis(np.linspace(0.05, 0.95, len(pred_xy)))
    ax.scatter(pred_xy[:, 0], pred_xy[:, 1], s=34, c=colors, edgecolors="black", linewidths=0.4, zorder=4, label="Predicted checkpoints")

    for (x, y), r, c in zip(pred_xy, pred_unc, colors):
        if np.isfinite(r) and r > 0:
            circ = plt.Circle((x, y), r, fill=False, linestyle="--", linewidth=1.0, alpha=0.7, color=c)
            ax.add_patch(circ)

    # True full trajectory is drawn last/on top.
    try:
        pos_df = load_cached_ball_positions(scene).dropna(subset=["ball_x", "ball_y"]).reset_index(drop=True)
        full_true = pos_df.iloc[input_last_frame:final_frame + 1][["ball_x", "ball_y"]].to_numpy(dtype=float)
        if len(full_true) > 1:
            ax.plot(full_true[:, 0], full_true[:, 1], color="red", linewidth=2.3, alpha=0.95, label="True trajectory", zorder=6)
    except Exception:
        ax.plot(true_xy[:, 0], true_xy[:, 1], color="red", linewidth=2.3, alpha=0.95, label="True checkpoints", zorder=6)

    # Annotate a few offsets to keep the plot readable.
    for i, ((x, y), k) in enumerate(zip(pred_xy, offsets)):
        if i == 0 or i == len(pred_xy) - 1 or k % 50 == 0:
            ax.text(x + 4, y - 4, f"+{k}", fontsize=7, color="white",
                    bbox=dict(facecolor="black", alpha=0.45, edgecolor="none", pad=1.0), zorder=7)

    ax.set_xlim(0, ORIGINAL_FRAME_WIDTH)
    ax.set_ylim(ORIGINAL_FRAME_HEIGHT, 0)
    ax.set_title(f"{scene.name}: predicted checkpoints over true trajectory")
    ax.legend(loc="lower right", fontsize=8)
    ax.axis("off")
    plt.tight_layout()

    if out_path is not None:
        fig.savefig(out_path, dpi=150)
        plt.close(fig)
    else:
        plt.show()


def train_one_run(run_name, lambda_err=LAMBDA_ERR, overlay_stride=OVERLAY_STRIDE):
    checkpoint_mode = CHECKPOINT_MODE
    train_ds, test_ds, train_loader, test_loader, max_checkpoints, global_max_future_offset, checkpoint_plan_df = make_loaders_for_sliding_windows()

    run_outdir = TRIAL_ROOT / run_name
    run_outdir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 100)
    print(f"Training run: {run_name}")
    print(f"lambda={lambda_err}")
    print(f"checkpoint_mode={checkpoint_mode}")
    print(f"Run output directory: {run_outdir.resolve()}")
    print("=" * 100)

    model, optimizer, scaler = make_new_model_and_optimizer()

    print(model)
    print("Parameter count:", sum(p.numel() for p in model.parameters()))
    print("Model device:", next(model.parameters()).device)

    history = []
    start_time = time.time()

    best_test_total_loss = float("inf")
    best_epoch = None
    best_checkpoint_path = run_outdir / "best_int_checkpoint_rnn_model.pt"
    best_epoch_summary_path = run_outdir / "best_epoch_summary.json"
    live_history_path = run_outdir / "training_history_live.csv"

    for epoch in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()

        train_total = 0.0
        train_pos = 0.0
        train_err = 0.0
        train_batches = 0

        for X, Y, target_mask, checkpoint_offsets, checkpoint_times, sample_idx in train_loader:
            X = X.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)
            target_mask = target_mask.to(DEVICE, non_blocking=True)
            checkpoint_times = checkpoint_times.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast_context():
                pred_pos, pred_err_mag_raw = model(X, checkpoint_times)
                loss, pos_loss, err_loss = augmented_loss(Y, target_mask, pred_pos, pred_err_mag_raw, lambda_err)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

            train_total += loss.item()
            train_pos += pos_loss.item()
            train_err += err_loss.item()
            train_batches += 1

        model.eval()
        test_total = 0.0
        test_pos = 0.0
        test_err = 0.0
        test_batches = 0

        with torch.no_grad():
            for X, Y, target_mask, checkpoint_offsets, checkpoint_times, sample_idx in test_loader:
                X = X.to(DEVICE, non_blocking=True)
                Y = Y.to(DEVICE, non_blocking=True)
                target_mask = target_mask.to(DEVICE, non_blocking=True)
                checkpoint_times = checkpoint_times.to(DEVICE, non_blocking=True)

                with autocast_context():
                    pred_pos, pred_err_mag_raw = model(X, checkpoint_times)
                    loss, pos_loss, err_loss = augmented_loss(Y, target_mask, pred_pos, pred_err_mag_raw, lambda_err)

                test_total += loss.item()
                test_pos += pos_loss.item()
                test_err += err_loss.item()
                test_batches += 1

        if DEVICE == "cuda":
            torch.cuda.synchronize()

        epoch_time = time.time() - epoch_start
        elapsed = time.time() - start_time
        avg_epoch_time = elapsed / epoch
        remaining = avg_epoch_time * (EPOCHS - epoch)

        row = {
            "epoch": epoch,
            "lambda_err": lambda_err,
            "run_name": run_name,
            "checkpoint_mode": checkpoint_mode,
            "train_total_loss": train_total / train_batches,
            "train_position_loss": train_pos / train_batches,
            "train_error_loss": train_err / train_batches,
            "test_total_loss": test_total / test_batches,
            "test_position_loss": test_pos / test_batches,
            "test_error_loss": test_err / test_batches,
            "epoch_minutes": epoch_time / 60,
            "avg_epoch_minutes": avg_epoch_time / 60,
            "remaining_minutes": remaining / 60,
        }
        history.append(row)

        # Save live history after every epoch, so progress survives timeout/interruption.
        pd.DataFrame(history).to_csv(live_history_path, index=False)

        # Save the best-test-loss model immediately after each epoch.
        if row["test_total_loss"] < best_test_total_loss:
            best_test_total_loss = float(row["test_total_loss"])
            best_epoch = int(epoch)

            best_payload = {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler is not None else None,
                "epoch": best_epoch,
                "best_test_total_loss": best_test_total_loss,
                "row": row,
                "config": {
                    "N_HISTORY": N_HISTORY,
                    "CHECKPOINT_MODE": checkpoint_mode,
                    "CHECKPOINT_STRIDE": CHECKPOINT_STRIDE,
                    "OVERLAY_STRIDE": overlay_stride,
                    "MAX_CHECKPOINTS": max_checkpoints,
                    "GLOBAL_MAX_FUTURE_OFFSET": global_max_future_offset,
                    "IMAGE_H": IMAGE_H,
                    "IMAGE_W": IMAGE_W,
                    "FRAME_CACHE_DIR": str(FRAME_CACHE_DIR),
                    "BALL_CACHE_DIR": str(BALL_CACHE_DIR),
                    "TRAIN_DATA_ROOT": str(TRAIN_DATA_ROOT),
                    "TEST_DATA_ROOT": str(TEST_DATA_ROOT),
                    "HIDDEN_CHANNELS": HIDDEN_CHANNELS,
                    "TIME_EMBED_DIM": TIME_EMBED_DIM,
                    "LAMBDA_ERR": lambda_err,
                    "WINDOWS_PER_SCENE_PER_EPOCH": WINDOWS_PER_SCENE_PER_EPOCH,
                    "EPOCHS": EPOCHS,
                    "BATCH_SIZE": BATCH_SIZE,
                    "LR": LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "offset0_current_position_target_included": True,
                    "scene_balanced_loss": True,
                    "input_window_policy": "training uses random consecutive 15-frame windows wholly before stop_event_frame; testing uses first 15 frames only",
                    "input_window_entirely_nonfreeze": True,
                    "targets_continue_through_final_frozen_frame": True,
                    "TEST_USE_FIRST_HISTORY_ONLY": TEST_USE_FIRST_HISTORY_ONLY,
                    "TEST_WINDOW_STRIDE": TEST_WINDOW_STRIDE,
                    "train_scenes": [str(path) for path in train_scene_dirs],
                    "test_scenes": [str(path) for path in test_scene_dirs],
                    "model_class": "InTCheckpointRNN",
                    "ORIGINAL_FRAME_WIDTH": ORIGINAL_FRAME_WIDTH,
                    "ORIGINAL_FRAME_HEIGHT": ORIGINAL_FRAME_HEIGHT,
                    "COORD_SCALE": COORD_SCALE.tolist(),
                },
            }
            torch.save(best_payload, best_checkpoint_path)

            with open(best_epoch_summary_path, "w") as f:
                json.dump({
                    "best_epoch": best_epoch,
                    "best_test_total_loss": best_test_total_loss,
                    "row": row,
                    "best_checkpoint_path": str(best_checkpoint_path),
                }, f, indent=2)

            print(
                f"  New best model saved at epoch {best_epoch}: "
                f"test_total_loss={best_test_total_loss:.6f} -> {best_checkpoint_path}",
                flush=True,
            )

        if True:
            print(
                f"{run_name} | Epoch {epoch:03d}/{EPOCHS} | "
                f"train total {row['train_total_loss']:.5f} | "
                f"test total {row['test_total_loss']:.5f} | "
                f"test pos {row['test_position_loss']:.5f} | "
                f"test err {row['test_error_loss']:.5f} | "
                f"epoch {row['epoch_minutes']:.2f} min | "
                f"remain {row['remaining_minutes']:.1f} min",
                flush=True,
            )
            if DEVICE == "cuda":
                print(f"  GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

    history_df = pd.DataFrame(history)
    history_df.to_csv(run_outdir / "training_history.csv", index=False)

    plt.figure(figsize=(7, 4))
    plt.plot(history_df["epoch"], history_df["train_total_loss"], label="train total")
    plt.plot(history_df["epoch"], history_df["test_total_loss"], label="test total")
    plt.plot(history_df["epoch"], history_df["test_position_loss"], label="test position")
    plt.xlabel("epoch")
    plt.ylabel("loss")
    plt.title(f"{run_name}: lambda={lambda_err}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(run_outdir / "training_history.png", dpi=150)
    plt.close()

    # Archive the epoch-150 state separately.
    last_epoch_checkpoint_path = (
        run_outdir / "last_epoch_int_checkpoint_rnn_model.pt"
    )
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "epoch": EPOCHS,
            "config": {
                "N_HISTORY": N_HISTORY,
                "CHECKPOINT_MODE": checkpoint_mode,
                "CHECKPOINT_STRIDE": CHECKPOINT_STRIDE,
                "OVERLAY_STRIDE": overlay_stride,
                "MAX_CHECKPOINTS": max_checkpoints,
                "GLOBAL_MAX_FUTURE_OFFSET": global_max_future_offset,
                "IMAGE_H": IMAGE_H,
                "IMAGE_W": IMAGE_W,
                "FRAME_CACHE_DIR": str(FRAME_CACHE_DIR),
                "BALL_CACHE_DIR": str(BALL_CACHE_DIR),
                "TRAIN_DATA_ROOT": str(TRAIN_DATA_ROOT),
                "TEST_DATA_ROOT": str(TEST_DATA_ROOT),
                "HIDDEN_CHANNELS": HIDDEN_CHANNELS,
                "TIME_EMBED_DIM": TIME_EMBED_DIM,
                "LAMBDA_ERR": lambda_err,
                "error_head_target": (
                    "absolute_magnitude_error_image_size_normalized"
                ),
                "coordinate_normalization": "image_size",
                "scene_balanced_loss": True,
                "input_only_first_frames": False,
                "input_window_policy": (
                    "training uses WINDOWS_PER_SCENE_PER_EPOCH random "
                    "consecutive 15-frame windows wholly before "
                    "stop_event_frame; testing uses first 15 frames only"
                ),
                "TRAIN_RANDOM_WINDOW_PER_SCENE_PER_EPOCH": (
                    TRAIN_RANDOM_WINDOW_PER_SCENE_PER_EPOCH
                ),
                "WINDOWS_PER_SCENE_PER_EPOCH": (
                    WINDOWS_PER_SCENE_PER_EPOCH
                ),
                "TEST_USE_FIRST_HISTORY_ONLY": (
                    TEST_USE_FIRST_HISTORY_ONLY
                ),
                "TEST_WINDOW_STRIDE": TEST_WINDOW_STRIDE,
                "model_class": "InTCheckpointRNN",
                "ORIGINAL_FRAME_WIDTH": ORIGINAL_FRAME_WIDTH,
                "ORIGINAL_FRAME_HEIGHT": ORIGINAL_FRAME_HEIGHT,
                "COORD_SCALE": COORD_SCALE.tolist(),
                "train_scenes": [str(path) for path in train_scene_dirs],
                "test_scenes": [str(path) for path in test_scene_dirs],
            },
        },
        last_epoch_checkpoint_path,
    )

    if best_epoch is None or not best_checkpoint_path.exists():
        raise RuntimeError("No best-model checkpoint was saved.")

    # Final evaluation, predictions, overlays, and canonical model file use
    # the best held-out-loss epoch rather than automatically using epoch 150.
    best_payload = torch.load(
        best_checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(best_payload["model_state_dict"])
    model.eval()

    canonical_best_model_path = (
        run_outdir / "int_checkpoint_rnn_model.pt"
    )
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "best_epoch": int(best_epoch),
            "best_test_total_loss": float(best_test_total_loss),
            "source_best_checkpoint": str(best_checkpoint_path),
            "config": best_payload["config"],
        },
        canonical_best_model_path,
    )

    print(
        f"Loaded best epoch {best_epoch} for final evaluation: "
        f"test_total_loss={best_test_total_loss:.6f}",
        flush=True,
    )

    # ----------------------------
    # Evaluate and save predictions
    # ----------------------------
    model.eval()
    prediction_rows = []

    with torch.no_grad():
        for X, Y, target_mask, checkpoint_offsets, checkpoint_times, sample_idx in test_loader:
            X = X.to(DEVICE, non_blocking=True)
            Y = Y.to(DEVICE, non_blocking=True)
            target_mask = target_mask.to(DEVICE, non_blocking=True)
            checkpoint_times = checkpoint_times.to(DEVICE, non_blocking=True)

            with autocast_context():
                pred_pos, pred_err_mag_raw = model(X, checkpoint_times)

            pred_pos_np = pred_pos.float().cpu().numpy()
            pred_err_mag_norm_np = positive_error_magnitude(pred_err_mag_raw).float().cpu().numpy()
            true_np = Y.float().cpu().numpy()
            mask_np = target_mask.float().cpu().numpy()
            offsets_np = checkpoint_offsets.cpu().numpy() if torch.is_tensor(checkpoint_offsets) else np.asarray(checkpoint_offsets)
            times_np = checkpoint_times.float().cpu().numpy()

            pred_pos_pix = unnormalize_y(pred_pos_np)
            true_pix = unnormalize_y(true_np)
            pred_err_mag_pix_approx = pred_err_mag_norm_np * ERR_MAG_PIXEL_SCALE_FOR_PLOTS

            sample_idx_np = sample_idx.cpu().numpy() if torch.is_tensor(sample_idx) else np.asarray(sample_idx)
            batch_size = pred_pos_pix.shape[0]

            for b in range(batch_size):
                meta = test_ds.get_meta(int(sample_idx_np[b]))
                scene_path = meta["scene"]
                start = int(meta["start"])
                input_last_frame = int(meta["input_last_frame"])
                final_frame = int(meta["final_frame"])
                n_checkpoints = int(meta["n_checkpoints"])

                valid_len = int(mask_np[b].sum())
                valid_len = min(valid_len, n_checkpoints, pred_pos_pix.shape[1])

                for j in range(valid_len):
                    checkpoint_offset = int(offsets_np[b, j])
                    true_xy = true_pix[b, j]
                    pred_xy = pred_pos_pix[b, j]
                    actual_error = true_xy - pred_xy
                    predicted_error_mag_norm = pred_err_mag_norm_np[b, j]
                    predicted_error_mag_pix_approx = pred_err_mag_pix_approx[b, j]
                    target_frame = input_last_frame + checkpoint_offset
                    is_final = bool(target_frame == final_frame)
                    shown = bool((checkpoint_offset % overlay_stride == 0) or is_final)

                    prediction_rows.append({
                        "lambda_err": lambda_err,
                        "run_name": run_name,
                        "checkpoint_mode": checkpoint_mode,
                        "scene": scene_path,
                        "scene_name": Path(scene_path).name,
                        "start": start,
                        "input_last_frame": input_last_frame,
                        "final_frame": final_frame,
                        "n_checkpoints": n_checkpoints,
                        "checkpoint_index": j,
                        "checkpoint_offset": checkpoint_offset,
                        "checkpoint_time_norm": float(times_np[b, j]),
                        "horizon_k": checkpoint_offset,
                        "target_frame": target_frame,
                        "is_final_frame_checkpoint": is_final,
                        "shown_in_overlay": shown,
                        "true_x": float(true_xy[0]),
                        "true_y": float(true_xy[1]),
                        "pred_x": float(pred_xy[0]),
                        "pred_y": float(pred_xy[1]),
                        "actual_error_x": float(actual_error[0]),
                        "actual_error_y": float(actual_error[1]),
                        "euclidean_error": float(np.linalg.norm(actual_error)),
                        "true_error_magnitude_normalized": float(np.linalg.norm(actual_error / COORD_SCALE)),
                        "predicted_error_magnitude_normalized": float(predicted_error_mag_norm),
                        "predicted_error_magnitude_pixels_approx": float(predicted_error_mag_pix_approx),
                        "predicted_uncertainty": float(predicted_error_mag_pix_approx),
                    })

    pred_df = pd.DataFrame(prediction_rows)
    pred_df.to_csv(run_outdir / "test_predictions.csv", index=False)

    summary = {
        "lambda_err": lambda_err,
        "run_name": run_name,
        "checkpoint_mode": checkpoint_mode,
        "best_epoch": int(best_epoch),
        "best_test_total_loss": float(best_test_total_loss),
        "best_checkpoint_path": str(best_checkpoint_path),
        "canonical_best_model_path": str(canonical_best_model_path),
        "last_epoch_checkpoint_path": str(last_epoch_checkpoint_path),
        "final_predictions_use_best_epoch": True,
        "mean_test_euclidean_error_pixels": float(pred_df["euclidean_error"].mean()),
        "median_test_euclidean_error_pixels": float(pred_df["euclidean_error"].median()),
        "max_test_euclidean_error_pixels": float(pred_df["euclidean_error"].max()),
        "mean_predicted_uncertainty_pixels_approx": float(pred_df["predicted_uncertainty"].mean()),
        "mean_true_error_magnitude_normalized": float(pred_df["true_error_magnitude_normalized"].mean()),
        "mean_predicted_error_magnitude_normalized": float(pred_df["predicted_error_magnitude_normalized"].mean()),
        "error_head_target": "absolute_magnitude_error_image_size_normalized",
        "coordinate_normalization": "image_size",
        "scene_balanced_loss": True,
        "input_frames": N_HISTORY,
        "training_targets": "offset0_current_position_plus_every_future_point_through_final_frozen_frame",
        "input_window_entirely_nonfreeze": True,
        "target_includes_post_event_freeze": True,
        "input_window_policy": "random 15-frame non-freeze windows for training; first15_only_for_testing",
        "offset0_current_position_target_included": True,
        "windows_per_scene_per_epoch": WINDOWS_PER_SCENE_PER_EPOCH,
        "test_use_first_history_only": TEST_USE_FIRST_HISTORY_ONLY,
        "test_window_stride": TEST_WINDOW_STRIDE,
        "overlay_stride": overlay_stride,
        "max_checkpoints": max_checkpoints,
        "global_max_future_offset": global_max_future_offset,
        "n_prediction_rows": int(len(pred_df)),
        "train_scene_count": len(train_scene_dirs),
        "test_scene_count": len(test_scene_dirs),
        "train_samples_per_epoch": len(train_ds),
        "test_first15_samples": len(test_ds),
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "lr": LR,
        "weight_decay": WEIGHT_DECAY,
        "image_h": IMAGE_H,
        "image_w": IMAGE_W,
        "hidden_channels": HIDDEN_CHANNELS,
        "time_embed_dim": TIME_EMBED_DIM,
        "frame_cache_dir": str(FRAME_CACHE_DIR),
        "ball_cache_dir": str(BALL_CACHE_DIR),
        "train_data_root": str(TRAIN_DATA_ROOT),
    }

    with open(run_outdir / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)
    pd.DataFrame([summary]).to_csv(run_outdir / "summary.csv", index=False)

    horizon_df = (
        pred_df
        .groupby("checkpoint_offset")[[
            "true_error_magnitude_normalized",
            "predicted_error_magnitude_normalized",
            "euclidean_error",
            "predicted_uncertainty",
        ]]
        .mean()
        .reset_index()
    )
    horizon_df.to_csv(run_outdir / "test_error_by_checkpoint_offset.csv", index=False)

    overlay_horizon_df = horizon_df[horizon_df["checkpoint_offset"] % overlay_stride == 0].copy()
    overlay_horizon_df.to_csv(run_outdir / "test_error_by_overlay_checkpoint_offset.csv", index=False)

    plt.figure(figsize=(7, 4))
    plt.plot(horizon_df["checkpoint_offset"], horizon_df["euclidean_error"], linewidth=1.5, label="true pixel error")
    plt.plot(horizon_df["checkpoint_offset"], horizon_df["predicted_uncertainty"], linewidth=1.5, label="predicted approx pixel radius")
    plt.xlabel("future offset from last input frame")
    plt.ylabel("pixels")
    plt.title(f"{run_name}: test error/uncertainty by future offset")
    plt.legend()
    plt.tight_layout()
    plt.savefig(run_outdir / "test_error_uncertainty_by_future_offset.png", dpi=150)
    plt.close()

    # One overlay per test scene. Testing uses first 15 frames only; for everypoint mode, show only every 10th prediction plus final.
    overlay_dir = run_outdir / "prediction_overlay_plots"
    if overlay_dir.exists():
        shutil.rmtree(overlay_dir)
    overlay_dir.mkdir(parents=True, exist_ok=True)

    windows = pred_df[["scene", "start", "input_last_frame"]].drop_duplicates().sort_values("scene")
    if N_OVERLAY_PLOTS is not None:
        n_plots = min(int(N_OVERLAY_PLOTS), len(windows))
        windows = windows.sample(n=n_plots, random_state=SEED)
    else:
        n_plots = len(windows)

    for i, (_, w) in enumerate(windows.iterrows()):
        group = pred_df[
            (pred_df["scene"] == w["scene"]) &
            (pred_df["start"] == w["start"]) &
            (pred_df["input_last_frame"] == w["input_last_frame"])
        ].copy()
        group_to_plot = group[group["shown_in_overlay"]].copy()
        if len(group_to_plot) == 0:
            group_to_plot = group.tail(1).copy()

        out_path = overlay_dir / f"overlay_{i:03d}_{Path(w['scene']).name}_{checkpoint_mode}_every10shown.png"
        plot_prediction_overlay(group_to_plot, out_path=out_path)

    print(f"Saved {n_plots} overlay plots to {overlay_dir}")
    print("Summary:")
    print(json.dumps(summary, indent=2))

    del model, optimizer, scaler
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return summary


In [ ]:
# ============================================================
# 7. Run 15-frame non-freeze sliding-window training condition
# ============================================================
# One run only:
#   - input = random consecutive 15-frame windows wholly before freeze
#   - training = 20 random windows per scene per epoch
#   - targets = offset 0 plus every frame through the final frozen frame
#   - lambda = 0.2
#   - epochs = 150
#   - best held-out-loss model is saved immediately after every improvement
# ============================================================

RUN_NAME = (
    "lambda_0_2_offset0_everypoint_"
    "sliding15_nonfreeze_windows20_epochs150"
)

summary_everypoint_sliding = train_one_run(
    run_name=RUN_NAME,
    lambda_err=LAMBDA_ERR,
    overlay_stride=OVERLAY_STRIDE,
)

all_run_summary_df = pd.DataFrame(
    [summary_everypoint_sliding]
)
all_run_summary_df.to_csv(
    TRIAL_ROOT / "all_run_summary.csv",
    index=False,
)

print("\nRun complete.")
print(
    "Summary saved to:",
    TRIAL_ROOT / "all_run_summary.csv",
)
display(all_run_summary_df)


In [ ]:
# ============================================================
# Error-head diagnostics using every 25th point in each test scene
# Plots:
#   1. Distribution of true error, log x-axis
#   2. Distribution of predicted error, log x-axis
#   3. Log-log scatter of predicted vs true error
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EVERY_N = 100

# ----------------------------
# Locate the current sliding-15 prediction CSV
# ----------------------------
RUN_DIR = TRIAL_ROOT / RUN_NAME
PRED_CSV = RUN_DIR / "test_predictions.csv"

if not PRED_CSV.exists():
    matches = sorted(TRIAL_ROOT.glob("**/test_predictions.csv"))
    if not matches:
        raise FileNotFoundError(
            f"Could not find test_predictions.csv under {TRIAL_ROOT}"
        )
    PRED_CSV = matches[-1]
    RUN_DIR = PRED_CSV.parent

print("Using prediction file:", PRED_CSV)

df = pd.read_csv(PRED_CSV)
print("Original rows:", len(df))
print("Columns:", list(df.columns))

# ----------------------------
# Pick columns robustly
# ----------------------------
scene_col_candidates = ["scene", "scene_name", "scene_dir", "test_scene", "scene_path"]
frame_col_candidates = ["frame", "frame_idx", "frame_index", "target_frame", "target_idx", "future_frame"]

scene_col = next((c for c in scene_col_candidates if c in df.columns), None)
frame_col = next((c for c in frame_col_candidates if c in df.columns), None)

if scene_col is None:
    raise KeyError(f"Could not find a scene column. Tried: {scene_col_candidates}")

if frame_col is None:
    print("WARNING: Could not find frame column. Will use row order within each scene.")
else:
    print("Using scene column:", scene_col)
    print("Using frame column:", frame_col)

if "euclidean_error" in df.columns:
    true_col = "euclidean_error"
    true_label = "True pixel error"
elif "true_error_magnitude_pixels" in df.columns:
    true_col = "true_error_magnitude_pixels"
    true_label = "True pixel error"
elif "true_error_magnitude_normalized" in df.columns:
    true_col = "true_error_magnitude_normalized"
    true_label = "True normalized error"
else:
    raise KeyError("Could not find a true error column.")

if "predicted_uncertainty" in df.columns:
    pred_col = "predicted_uncertainty"
    pred_label = "Predicted pixel error"
elif "predicted_error_magnitude_pixels_approx" in df.columns:
    pred_col = "predicted_error_magnitude_pixels_approx"
    pred_label = "Predicted pixel error"
elif "predicted_error_magnitude_normalized" in df.columns:
    pred_col = "predicted_error_magnitude_normalized"
    pred_label = "Predicted normalized error"
else:
    raise KeyError("Could not find a predicted error column.")

print("Using true error column:", true_col)
print("Using predicted error column:", pred_col)

# ----------------------------
# Clean and sort
# ----------------------------
use_cols = [scene_col, true_col, pred_col]
if frame_col is not None:
    use_cols.append(frame_col)

plot_df = df[use_cols].copy()
plot_df[true_col] = pd.to_numeric(plot_df[true_col], errors="coerce")
plot_df[pred_col] = pd.to_numeric(plot_df[pred_col], errors="coerce")

if frame_col is not None:
    plot_df[frame_col] = pd.to_numeric(plot_df[frame_col], errors="coerce")
    plot_df = plot_df.sort_values([scene_col, frame_col])
else:
    plot_df = plot_df.sort_values([scene_col])

plot_df = (
    plot_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[scene_col, true_col, pred_col])
    .reset_index(drop=True)
)

# ----------------------------
# Keep every 25th point within each test scene
# ----------------------------
# Keeps rows with within-scene index 0, 25, 50, 75, ...
plot_df["within_scene_point_index"] = plot_df.groupby(scene_col).cumcount()
plot_df_n = plot_df[plot_df["within_scene_point_index"] % EVERY_N == 0].copy()

plot_df_n = plot_df_n.rename(columns={
    true_col: "true_error",
    pred_col: "predicted_error",
})

print("Clean rows:", len(plot_df))
print(f"Rows after keeping every {EVERY_N}th point per scene:", len(plot_df_n))
print("Number of scenes:", plot_df_n[scene_col].nunique())

# ----------------------------
# Positive-only version for log plots
# ----------------------------
log_df = plot_df_n[
    (plot_df_n["true_error"] > 0) &
    (plot_df_n["predicted_error"] > 0)
].copy()

print("Rows usable for log plots:", len(log_df))

pearson_raw = plot_df_n["true_error"].corr(plot_df_n["predicted_error"], method="pearson")
spearman_raw = plot_df_n["true_error"].corr(plot_df_n["predicted_error"], method="spearman")

pearson_log = np.log10(log_df["true_error"]).corr(
    np.log10(log_df["predicted_error"]),
    method="pearson",
)
spearman_log = log_df["true_error"].corr(
    log_df["predicted_error"],
    method="spearman",
)

print(f"Pearson r, raw values:    {pearson_raw:.4f}")
print(f"Spearman r, raw values:   {spearman_raw:.4f}")
print(f"Pearson r, log10 values:  {pearson_log:.4f}")
print(f"Spearman r, log-log data: {spearman_log:.4f}")

# ----------------------------
# Save folder
# ----------------------------
DIAG_DIR = RUN_DIR / f"error_head_diagnostics_every{EVERY_N}"
DIAG_DIR.mkdir(parents=True, exist_ok=True)

filtered_csv = DIAG_DIR / f"test_predictions_every{EVERY_N}_error_diagnostics.csv"
plot_df_n.to_csv(filtered_csv, index=False)
print(f"Saved filtered every-{EVERY_N} data to:", filtered_csv)

# ----------------------------
# Helper: log-spaced bins
# ----------------------------
def make_log_bins(values, n_bins=80):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    if len(values) == 0:
        raise ValueError("No positive finite values available for log bins.")
    vmin = values.min()
    vmax = values.max()
    if vmin == vmax:
        vmin = vmin * 0.9
        vmax = vmax * 1.1
    return np.logspace(np.log10(vmin), np.log10(vmax), n_bins)

# ----------------------------
# 1. Distribution of true error, log x-axis
# ----------------------------
true_positive = log_df["true_error"].to_numpy()
true_bins = make_log_bins(true_positive, n_bins=80)

plt.figure(figsize=(7, 4))
plt.hist(true_positive, bins=true_bins, alpha=0.85)
plt.xscale("log")
plt.xlabel(true_label + " (log scale)")
plt.ylabel("Count")
plt.title(f"Distribution of true error\nEvery {EVERY_N}th point per test scene")
plt.tight_layout()
plt.savefig(DIAG_DIR / f"distribution_true_error_every{EVERY_N}_logx.png", dpi=150)
plt.show()

# ----------------------------
# 2. Distribution of predicted error, log x-axis
# ----------------------------
pred_positive = log_df["predicted_error"].to_numpy()
pred_bins = make_log_bins(pred_positive, n_bins=80)

plt.figure(figsize=(7, 4))
plt.hist(pred_positive, bins=pred_bins, alpha=0.85)
plt.xscale("log")
plt.xlabel(pred_label + " (log scale)")
plt.ylabel("Count")
plt.title(f"Distribution of predicted error\nEvery {EVERY_N}th point per test scene")
plt.tight_layout()
plt.savefig(DIAG_DIR / f"distribution_predicted_error_every{EVERY_N}_logx.png", dpi=150)
plt.show()

# ----------------------------
# 3. Scatterplot: true error vs predicted error, log-log axes
# ----------------------------
plt.figure(figsize=(6, 6))
plt.scatter(
    log_df["true_error"],
    log_df["predicted_error"],
    s=10,
    alpha=0.35,
)

plt.xscale("log")
plt.yscale("log")

min_val = float(np.nanmin([
    log_df["true_error"].min(),
    log_df["predicted_error"].min(),
]))
max_val = float(np.nanmax([
    log_df["true_error"].max(),
    log_df["predicted_error"].max(),
]))
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--", linewidth=1)

plt.xlabel(true_label + " (log scale)")
plt.ylabel(pred_label + " (log scale)")
plt.title(
    "Predicted vs true error, log-log\n"
    f"Every {EVERY_N}th point per scene; Pearson log10 r={pearson_log:.3f}, Spearman r={spearman_log:.3f}"
)
plt.tight_layout()
plt.savefig(DIAG_DIR / f"scatter_true_vs_predicted_error_every{EVERY_N}_loglog.png", dpi=150)
plt.show()

print("Saved plots to:", DIAG_DIR)